# Kaggle GPU — VietOCR FT inference with Beam Search

No retraining. Loads existing `vietocr_ft.pth` weights (from the original fine-tune run)
and re-runs inference with `beamsearch=True`. Outputs `ocr_vietocr_ftbs_{test,all}.parquet`.

## Setup
1. New notebook inside the competition, **GPU T4**, **Internet ON**.
2. Attach the dataset containing `vietocr_ft.pth` (the weights saved from the original fine-tune).
   - If saved as a Kaggle dataset: Add Input → Your Datasets → select it.
   - Update `WEIGHTS_PATH` in Cell 3 to point to the correct path.
3. Run Cell 1 → **Restart & clear outputs** → Run All.
4. Use **Save Version → Save & Run All (Commit)** for the full run (~3h inference).
5. Download `ocr_vietocr_ftbs_test.parquet` + `_all` into local `cache/`.

In [ ]:
# Install. Restart kernel after this cell.
!pip install -q easyocr vietocr rapidfuzz
!pip install -q --force-reinstall 'Pillow==10.4.0'
import PIL, torch
print('Pillow', PIL.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
import re, zipfile, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image
from tqdm.auto import tqdm

KIN = Path('/kaggle/input'); ROOT = None
for p in KIN.rglob('test.csv'):
    ROOT = p.parent; break
assert ROOT is not None, 'competition data not mounted'
WORK = Path('/kaggle/working'); print('ROOT =', ROOT)

def _imgs(kind):
    want_train = (kind == 'train')
    for d in KIN.rglob('*'):
        if d.is_dir() and any(d.glob('img_*.jpg')):
            if ('train' in str(d).lower()) == want_train: return d
    for z in KIN.rglob('*.zip'):
        if ('train' in z.name.lower()) != want_train: continue
        ex = WORK/(kind+'_imgs'); ex.mkdir(parents=True, exist_ok=True)
        if not any(ex.rglob('img_*.jpg')):
            with zipfile.ZipFile(z) as zf: zf.extractall(ex)
        for d in [ex,*[c for c in ex.rglob('*') if c.is_dir()]]:
            if any(d.glob('img_*.jpg')): return d
    raise FileNotFoundError(kind)

TRAIN_DIR = _imgs('train'); TEST_DIR = _imgs('test')
labels = pd.read_csv(ROOT/'train_labels.csv', keep_default_na=False)
labels['ocr_text'] = labels['ocr_text'].astype(str).str.strip()
test_ids = pd.read_csv(ROOT/'test.csv')['image_id'].tolist()
print('train', len(list(TRAIN_DIR.glob('*.jpg'))), '| test', len(list(TEST_DIR.glob('*.jpg'))))

def clean_ocr(t, maxlen=500):
    t = unicodedata.normalize('NFC', str(t)).replace(chr(10),' ').replace(chr(9),' ')
    t = re.sub(r'\s+',' ',t).strip(); toks=t.split(); ded=[toks[0]] if toks else []
    for w in toks[1:]:
        if w.lower()!=ded[-1].lower(): ded.append(w)
    t=' '.join(ded); return t[:maxlen].rstrip() if len(t)>maxlen else t

In [ ]:
import easyocr
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

# ---- Point this at your saved weights ----
# If you saved as a Kaggle dataset, the path will be something like:
# /kaggle/input/ura-vietocr-weights/vietocr_ft.pth
# Check: !find /kaggle/input -name 'vietocr_ft.pth'
WEIGHTS_PATH = None  # auto-find below
for p in KIN.rglob('vietocr_ft.pth'):
    WEIGHTS_PATH = str(p); break
if WEIGHTS_PATH is None:
    # fallback: weights saved as output of original fine-tune notebook version
    for p in KIN.rglob('*.pth'):
        if 'ft' in p.name.lower(): WEIGHTS_PATH = str(p); break
assert WEIGHTS_PATH is not None, 'vietocr_ft.pth not found — attach the weights dataset'
print('weights:', WEIGHTS_PATH)

detector = easyocr.Reader(['vi','en'], gpu=True, verbose=False)
def detect_boxes(img):
    try:
        horiz,_ = detector.detect(img, min_size=15, text_threshold=0.6)
        return [tuple(int(v) for v in b) for b in (horiz[0] if horiz else [])]
    except Exception: return []

ft_cfg = Cfg.load_config_from_name('vgg_transformer')
ft_cfg['device'] = 'cuda:0'
ft_cfg['weights'] = WEIGHTS_PATH
ft_cfg['predictor']['beamsearch'] = True   # <-- THE ONLY CHANGE vs original inference
ft_rec = Predictor(ft_cfg)
print('loaded weights with beamsearch=True')

In [ ]:
def ocr_image(path):
    img = cv2.imread(str(path))
    if img is None: return {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}
    items=[]
    for (x0,x1,y0,y1) in detect_boxes(img):
        x0,y0=max(0,x0),max(0,y0); x1,y1=min(img.shape[1],x1),min(img.shape[0],y1)
        if x1-x0<6 or y1-y0<6: continue
        items.append((y0,x0,Image.fromarray(cv2.cvtColor(img[y0:y1,x0:x1],cv2.COLOR_BGR2RGB))))
    if not items: return {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}
    # beamsearch=True -> predict one at a time (batch not reliable with beam)
    texts = [ft_rec.predict(it[2]) for it in items]
    ys=[it[0] for it in items]; band=max(8.0,(max(ys)-min(ys))/40.0)
    order=sorted(range(len(items)), key=lambda i:(round(items[i][0]/band),items[i][1]))
    text=clean_ocr(' '.join(texts[i] for i in order if str(texts[i]).strip()))
    return {'ocr_text':text,'raw_text':text,'mean_conf':0.0,'n_boxes':len(items),'n_chars':len(text)}

def run(ids, img_dir, out, save_every=200):
    done={}
    if Path(out).exists(): done={r['image_id']:r for r in pd.read_parquet(out).to_dict('records')}
    recs=list(done.values()); pend=[i for i in ids if i not in done]
    print(out, 'pending', len(pend), '/', len(ids))
    for k,iid in enumerate(tqdm(pend)):
        r=ocr_image(img_dir/(iid+'.jpg')); r['image_id']=iid; recs.append(r)
        if (k+1)%save_every==0: pd.DataFrame(recs).to_parquet(out,index=False)
    pd.DataFrame(recs).to_parquet(out,index=False)
    print('saved', out, '| fill', (pd.DataFrame(recs).ocr_text.str.strip()!='').mean())

run(test_ids, TEST_DIR, '/kaggle/working/ocr_vietocr_ftbs_test.parquet')
run(labels['image_id'].tolist(), TRAIN_DIR, '/kaggle/working/ocr_vietocr_ftbs_all.parquet')
print('\nDONE — download ocr_vietocr_ftbs_test.parquet (+ _all).')